# GNNocRoute-DRL: Training + Energy + PARSEC
## GPU-Accelerated Training on Google Colab
---
**Author:** Ngọc Anh for Thầy Hiếu

**Instructions:** Runtime → Run all (hoặc Ctrl+F9)

**Time:** ~5-10 phút (GPU T4)

**Output:** Model + figures sẽ lưu về Google Drive

In [ ]:
# === SETUP ===
import os, sys, json, time, random, math, subprocess, re
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
from google.colab import drive

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

# Mount Drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/GNNocRoute_DRL_Results'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output: {OUT_DIR}')

In [ ]:
# === 1. GNN ENCODER (GATv2) ===
class GNNEncoder(nn.Module):
    """Graph Attention Network v2 encoder for NoC topology."""
    def __init__(self, in_dim=4, hidden=64, out=32, heads=4):
        super().__init__()
        from torch_geometric.nn import GATv2Conv
        self.conv1 = GATv2Conv(in_dim, hidden//heads, heads=heads)
        self.conv2 = GATv2Conv(hidden, out, heads=1)
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(out)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index); x = self.norm1(x); x = F.elu(x)
        x = self.conv2(x, edge_index); x = self.norm2(x)
        return x

# === DQN AGENT ===
class DQN(nn.Module):
    def __init__(self, in_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.LayerNorm(128),
            nn.Linear(128, 64), nn.ReLU(), nn.LayerNorm(64),
            nn.Linear(64, 4))
    def forward(self, x): return self.net(x)

print('✅ Models defined')
print(f'  GNN params: {sum(p.numel() for p in GNNEncoder().parameters()):,}')
print(f'  DQN params: {sum(p.numel() for p in DQN().parameters()):,}')

In [ ]:
# === 2. NoC ROUTING ENVIRONMENT ===
class NoCRoutingEnv:
    """Proper NoC routing simulation environment for DRL."""
    
    PARSEC_PROFILES = {
        'blackscholes': {'hotspot': 0.02, 'locality': 0.3, 'burst': 0.1},
        'bodytrack':    {'hotspot': 0.05, 'locality': 0.5, 'burst': 0.2},
        'fluidanimate': {'hotspot': 0.03, 'locality': 0.7, 'burst': 0.05},
        'canneal':      {'hotspot': 0.08, 'locality': 0.1, 'burst': 0.3},
    }
    
    def __init__(self, grid=4, traffic='hotspot', inj_rate=0.1, profile='blackscholes'):
        self.G = grid
        self.N = grid * grid
        self.inj = inj_rate
        self.traffic = traffic
        self.prof = self.PARSEC_PROFILES.get(profile, {'hotspot':0.05,'locality':0.3,'burst':0.1})
        
        # Build mesh edges
        self.edge_index = self._build_mesh()
        
        # State
        self.congestion = np.zeros(self.N)
        self.link_load = np.zeros((self.N, 4))
        self.packets = []
        self.steps = 0
        self.hotspot = self.G * (self.G // 2) + (self.G // 2)
    
    def _build_mesh(self):
        edges = []
        for y in range(self.G):
            for x in range(self.G):
                i = y * self.G + x
                if x > 0: edges.append([i, y*self.G+(x-1)])
                if x < self.G-1: edges.append([i, y*self.G+(x+1)])
                if y > 0: edges.append([i, (y-1)*self.G+x])
                if y < self.G-1: edges.append([i, (y+1)*self.G+x])
        return torch.LongTensor(edges).t()
    
    def _min_ports(self, src, dst):
        sx, sy = src % self.G, src // self.G
        dx, dy = dst % self.G, dst // self.G
        ports = []
        if dx > sx: ports.append(2)
        if dx < sx: ports.append(3)
        if dy > sy: ports.append(1)
        if dy < sy: ports.append(0)
        return ports if ports else [random.randint(0,3)]
    
    def step(self, dqn=None, gnn=None, epsilon=0):
        self.steps += 1
        
        # Generate packets
        for src in range(self.N):
            if random.random() < self.inj:
                if self.traffic == 'hotspot' and random.random() < 0.1:
                    dst = self.hotspot
                elif random.random() < self.prof['locality']:
                    sx, sy = src % self.G, src // self.G
                    dx = max(0, min(self.G-1, sx + random.randint(-1,1)))
                    dy = max(0, min(self.G-1, sy + random.randint(-1,1)))
                    dst = dy * self.G + dx
                else:
                    dst = random.randint(0, self.N-1)
                if dst != src:
                    self.packets.append({'src':src,'dst':dst,'pos':src,'hops':0})
        
        # Route
        delivered = 0
        lat_sum = 0
        new_pkts = []
        
        state = self._get_state()
        with torch.no_grad():
            if dqn and gnn:
                emb = gnn(state, self.edge_index)
                q = dqn(emb)
                actions = q.argmax(dim=1) if random.random() > epsilon else torch.randint(0,4,(self.N,))
            else:
                actions = None
        
        actions_taken = []
        for pkt in self.packets:
            p, d = pkt['pos'], pkt['dst']
            if p == d:
                delivered += 1; lat_sum += pkt['hops']; continue
            
            if actions is not None:
                port = actions[p].item()
            else:
                ports = self._min_ports(p, d)
                port = min(ports, key=lambda x: self.link_load[p,x])
            
            if self.congestion[p] > 10:
                new_pkts.append(pkt); continue
            
            actions_taken.append((p % self.N, port))
            x, y = p % self.G, p // self.G
            dx, dy = [(0,-1),(0,1),(1,0),(-1,0)][port]
            nx, ny = x+dx, y+dy
            
            if 0 <= nx < self.G and 0 <= ny < self.G:
                pkt['pos'] = ny*self.G+nx; pkt['hops'] += 1
                self.link_load[p,port] += 1
            else:
                # Fallback: minimal
                ports = self._min_ports(p, d)
                for pp in ports:
                    dx2,dy2 = [(0,-1),(0,1),(1,0),(-1,0)][pp]
                    nx2,ny2 = x+dx2, y+dy2
                    if 0 <= nx2 < self.G and 0 <= ny2 < self.G:
                        pkt['pos'] = ny2*self.G+nx2; pkt['hops'] += 1
                        break
            new_pkts.append(pkt)
        
        self.packets = new_pkts
        self.congestion = self.congestion * 0.9 + 0.1 * np.array(
            [len([p for p in self.packets if p['pos']==i]) for i in range(self.N)])
        self.link_load *= 0.9
        
        avg_lat = lat_sum / max(delivered, 1)
        reward = delivered * 0.5 - avg_lat * 0.05 - np.mean(self.congestion) * 0.3
        
        return self._get_state(), reward, self.steps >= 100, {'del':delivered,'lat':avg_lat}
    
    def _get_state(self):
        s = np.zeros((self.N, 4))
        for i in range(self.N):
            s[i,0] = self.congestion[i] / max(np.max(self.congestion), 1)
            s[i,1] = np.mean(self.link_load[i])
            s[i,2] = len([p for p in self.packets if p['pos']==i]) / 10
            ix, iy = i % self.G, i // self.G
            hx, hy = self.hotspot % self.G, self.hotspot // self.G
            s[i,3] = (abs(ix-hx)+abs(iy-hy)) / (2*self.G)
        return torch.FloatTensor(s)
    
    def reset(self):
        self.congestion = np.zeros(self.N)
        self.link_load = np.zeros((self.N, 4))
        self.packets = []
        self.steps = 0
        return self._get_state()

print('✅ NoC Routing Environment defined')

In [ ]:
# === 3. TRAINING LOOP (GPU) ===
print('=' * 60)
print('GNNocRoute-DRL: GPU Training')
print('=' * 60)

configs = [
    ('hotspot', 'blackscholes', 300),
    ('uniform', 'blackscholes', 200),
    ('hotspot', 'canneal', 200),
]

all_results = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (traffic, profile, episodes) in enumerate(configs):
    print(f'\n--- {idx+1}/{len(configs)}: {traffic}/{profile} ({episodes} eps) ---')
    
    gnn = GNNEncoder().to(DEVICE)
    dqn = DQN().to(DEVICE)
    target = DQN().to(DEVICE)
    target.load_state_dict(dqn.state_dict())
    opt = torch.optim.Adam(list(gnn.parameters())+list(dqn.parameters()), lr=5e-4)
    
    env = NoCRoutingEnv(traffic=traffic, profile=profile)
    memory = deque(maxlen=20000)
    rewards = []
    epsilon = 0.5
    
    t0 = time.time()
    for ep in range(episodes):
        state = env.reset()
        ep_r = 0
        for step in range(100):
            next_state, reward, done, info = env.step(dqn, gnn, epsilon)
            memory.append((state.cpu().numpy(), 0, reward, next_state.cpu().numpy(), done))
            state = next_state; ep_r += reward
            
            if len(memory) >= 64:
                batch = random.sample(list(memory), 64)
                ss = torch.FloatTensor(np.array([b[0] for b in batch])).to(DEVICE)
                aa = torch.LongTensor([b[1] for b in batch]).unsqueeze(1).to(DEVICE)
                rr = torch.FloatTensor([[b[2]] for b in batch]).to(DEVICE)
                nss = torch.FloatTensor(np.array([b[3] for b in batch])).to(DEVICE)
                
                with torch.no_grad():
                    tgt = rr + 0.95 * target(gnn(nss, env.edge_index)).max(1,keepdim=True)[0]
                current = dqn(gnn(ss, env.edge_index)).gather(1, aa)
                loss = F.mse_loss(current, tgt)
                opt.zero_grad(); loss.backward(); opt.step()
            if done: break
        
        epsilon = max(0.01, epsilon - 0.5/episodes)
        rewards.append(ep_r)
        if ep % 10 == 0: target.load_state_dict(dqn.state_dict())
        if (ep+1) % 100 == 0:
            print(f'  Ep {ep+1}/{episodes} | R={np.mean(rewards[-50:]):.1f} | ε={epsilon:.3f}')
    
    elapsed = time.time() - t0
    print(f'  ✅ Done in {elapsed/60:.1f} min | Avg reward: {np.mean(rewards[-50:]):.2f}')
    
    # Plot
    axes[idx].plot(rewards); axes[idx].set_title(f'{traffic}/{profile}')
    axes[idx].set_xlabel('Episode'); axes[idx].set_ylabel('Reward')
    axes[idx].grid(True, alpha=0.3)
    
    # Save model
    torch.save({'gnn':gnn.state_dict(),'dqn':dqn.state_dict()},
               f'{OUT_DIR}/model_{traffic}_{profile}.pt')
    all_results[f'{traffic}_{profile}'] = {
        'time_min': elapsed/60, 'avg_reward': float(np.mean(rewards[-50:])),
        'episodes': episodes, 'gpu': DEVICE}

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/training_curves.png', dpi=200)
print(f'\n✅ Saved: {OUT_DIR}/training_curves.png')

json.dump(all_results, open(f'{OUT_DIR}/results.json','w'), indent=2)
print(f'✅ Results saved to Drive!')

In [ ]:
# === 4. INSTALL BOOKSIM2 + ENERGY EXPERIMENTS ===
print('=== Installing BookSim2 with Power Model ===')
!git clone https://github.com/booksim/booksim2.git /content/booksim2 2>/dev/null || echo 'Already cloned'
%cd /content/booksim2/src
!make clean && make 2>&1 | tail -5

BOOKSIM = '/content/booksim2/src/booksim'
print(f'BookSim2: {"OK" if os.path.exists(BOOKSIM) else "FAILED"}')

# Run energy experiments
import tempfile, subprocess
outdir = f'{OUT_DIR}/booksim_energy'
os.makedirs(outdir, exist_ok=True)

summary = 'topology,algorithm,traffic,inj_rate,latency,energy\n'
count = 0
for topo in ['mesh44','mesh88']:
    k = 4 if '44' in topo else 8
    t = 'mesh'
    for algo in ['dor','adaptive_xy_yx','min_adapt']:
        for traffic in ['uniform','transpose','hotspot']:
            for inj in [0.01, 0.02, 0.05, 0.1]:
                count += 1
                cfg = tempfile.NamedTemporaryFile(mode='w', suffix='.cfg', delete=False)
                cfg.write(f'''
topology = {t}; k = {k}; n = 2;
routing_function = {algo};
num_vcs = 4; vc_buf_size = 8;
traffic = {traffic}; injection_rate = {inj};
sim_type = latency; warmup_periods = 3;
sample_period = 1000; sim_count = 1;
sim_power = 1;
tech_file = /content/booksim2/src/power/techfile.txt;
''')
                cfg.close()
                try:
                    r = subprocess.run([BOOkSIM, cfg.name], capture_output=True, text=True, timeout=30)
                    lat = re.search(r'Packet latency average\s*=\s*([0-9.]+)', r.stdout)
                    en = re.search(r'Total energy\s*=\s*([0-9.]+)', r.stdout)
                    lat_v = float(lat.group(1)) if lat else 'NA'
                    en_v = float(en.group(1)) if en else 'NA'
                    summary += f'{topo},{algo},{traffic},{inj},{lat_v},{en_v}\n'
                    print(f'  [{count}] {topo} {algo} {traffic} @{inj}: lat={lat_v} en={en_v}')
                except: summary += f'{topo},{algo},{traffic},{inj},NA,NA\n'
                os.unlink(cfg.name)

open(f'{outdir}/summary_energy.csv','w').write(summary)
print(f'\n✅ Energy experiments done: {count} configs')
print(f'Results: {outdir}/summary_energy.csv')

In [ ]:
# === 5. SUMMARY ===
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)

# DRL
results = json.load(open(f'{OUT_DIR}/results.json'))
for k,v in results.items():
    print(f'  DRL {k:30s}: {v["avg_reward"]:.2f} ({v["time_min"]:.1f} min)')

# Energy
if os.path.exists(f'{OUT_DIR}/booksim_energy/summary_energy.csv'):
    lines = open(f'{OUT_DIR}/booksim_energy/summary_energy.csv').read().strip().split('\n')
    print(f'  BookSim2 energy: {len(lines)-1} configs')

print(f'\nAll outputs in: {OUT_DIR}')
print(f'\n✅ GNNocRoute-DRL: Training + Energy + PARSEC complete!')